# Scenespy on Google Colab

A headless notebook to detect scenes, split videos by time interval, and extract faces without loading the graphical interface.

## 1. Download the project

In [ ]:
import os
import subprocess

repository = "https://github.com/guilhermejaques/scenespy.git"
project_dir = "/content/scenespy"

if os.path.isdir(os.path.join(project_dir, ".git")):
    subprocess.run(["git", "-C", project_dir, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", repository, project_dir], check=True)

os.chdir(project_dir)
print(project_dir)

## 2. Install the dependencies

The face pipeline uses MediaPipe, which requires NumPy 1.x and Protobuf 4.x. TensorFlow is preinstalled by Colab but is not used by Scenespy; it is removed because recent Colab TensorFlow builds require a newer, incompatible Protobuf. Use a separate Colab session for this notebook.

In [ ]:
%pip uninstall -q -y tensorflow
%pip install -q pillow==11.3.0 numpy==1.26.4 protobuf==4.25.8 opencv-contrib-python==4.11.0.86 av==16.1.0 scenedetect==0.6.7.1 mediapipe==0.10.21
%pip install -q --no-deps ultralytics==8.4.9

import os
print("Dependencies installed. Restarting the Colab session...")
os.kill(os.getpid(), 9)

The installation cell restarts the Colab session automatically. Wait for it to reconnect, then continue with the cell below without running the installation again.

## 3. Load and check the API

In [ ]:
import os
import sys

project_dir = "/content/scenespy"
os.chdir(project_dir)
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

import av
import cv2
import mediapipe
import numpy
import PIL
import scenedetect
import ultralytics
from importlib.metadata import version

from scenespy import __version__
from scenespy.api import detect_scenes, extract_faces, process_video, process_videos, split_video

expected = {
    "av": "16.1.0",
    "mediapipe": "0.10.21",
    "numpy": "1.26.4",
    "opencv-contrib-python": "4.11.0.86",
    "pillow": "11.3.0",
    "protobuf": "4.25.8",
    "scenedetect": "0.6.7.1",
    "ultralytics": "8.4.9",
}
installed = {package: version(package) for package in expected}
mismatches = {package: (expected[package], actual) for package, actual in installed.items() if actual != expected[package]}
if mismatches:
    raise RuntimeError(f"Dependency version mismatch: {mismatches}. Run the install cell and restart the session.")

print(f"Scenespy {__version__} loaded without the graphical interface.")
print(f"Pillow {PIL.__version__} | NumPy {numpy.__version__} | OpenCV {cv2.__version__} | MediaPipe {mediapipe.__version__} | Ultralytics {ultralytics.__version__}")

## 4. Select the accelerator

Enable a GPU in **Runtime → Change runtime type**. Use `nvidia` with an NVIDIA GPU or `cpu` without a GPU. If the selected accelerator is not available, the API safely uses the CPU.

In [ ]:
import torch

accelerator = "nvidia" if torch.cuda.is_available() else "cpu"
print(f"Accelerator: {accelerator}")

## 5A. Upload videos from your computer

Use this option for small files or quick tests. Uploaded files stay in the temporary session storage.

In [ ]:
from google.colab import files

uploaded = files.upload()
video_paths = [os.path.abspath(name) for name in uploaded]
if not video_paths:
    raise ValueError("No video was uploaded.")

video_path = video_paths[0]
print(video_paths)

## 5B. Use videos from Google Drive

Use this option if your videos are in Google Drive. Update the paths after mounting the drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
video_path = "/content/drive/MyDrive/videos/video.mp4"
video_paths = [video_path]
output_root = "/content/drive/MyDrive/scenespy_output"

## 6. Set the output folder

If you did not use the Google Drive option, results are temporarily saved in `/content/scenespy_output`.

In [ ]:
output_root = globals().get("output_root", "/content/scenespy_output")
os.makedirs(output_root, exist_ok=True)
print(output_root)

## 7A. Detect and cut scenes

Available sensitivity profiles: `Low`, `Normal`, `High`, and `Auto`.

In [ ]:
result = detect_scenes(
    video=video_path,
    output=output_root,
    sensitivity="Normal",
    accelerator=accelerator,
)
result

## 7B. Split by time interval

In [ ]:
result = split_video(
    video=video_path,
    output=output_root,
    interval=10,
    accelerator=accelerator,
)
result

## 7C. Detect and save faces

Available sensitivity profiles: `Low`, `Normal`, and `High`. The `Auto` profile is not available for face detection.

In [ ]:
result = extract_faces(
    video=video_path,
    output=output_root,
    sensitivity="Normal",
    accelerator=accelerator,
)
result

## 8. Process a batch

Available modes: `scene`, `interval`, and `faces`. By default, an invalid file is added to the results and does not stop the other files.

In [ ]:
results = process_videos(
    videos=video_paths,
    output=output_root,
    mode="faces",
    sensitivity="Normal",
    accelerator=accelerator,
    continue_on_error=True,
)
results

## 9. Download the results

This step creates a ZIP file from the output folder. Use it when the output is in `/content`. Files saved in Google Drive already remain there.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/scenespy_results", "zip", output_root)
files.download(archive)

## General API

You can also run every mode with `process_video`. The API checks paths, modes, sensitivity profiles, intervals, and accelerators before processing. Keep `verbose=True` to receive the same progress text used by the app.

In [ ]:
result = process_video(
    video=video_path,
    output=output_root,
    mode="scene",
    sensitivity="Normal",
    accelerator=accelerator,
    interval=10,
    verbose=True,
)
result